In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load Data and Feature Engineering

In [ ]:
# Define the Raw Data URLs
batterChallengeRatesURL = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSSituationalSplits-Batters-FullSeason.csv"
batterStatsURL = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-BattingStats.csv"
df_ChalRates = pd.read_csv(batterChallengeRatesURL, encoding="latin1")
df_BatterStats = pd.read_csv(batterStatsURL, encoding="latin1")

# Inner join the two DataFrames
df_merged = pd.merge(
  df_ChalRates,
  df_BatterStats,
  left_on="batterId",
  right_on="playerId",
  how="inner"
)
df_merged = df_merged.drop(columns=["playerId"])
df_merged.shape

# Create new rate stats
df_merged["strikeOutRate"] = df_merged["strikeOuts"] / df_merged["plateAppearances"]
df_merged["walkRate"] = df_merged["baseOnBalls"] / df_merged["plateAppearances"]
df_merged["pitchesPerPA"] = df_merged["numberOfPitches"] / df_merged["plateAppearances"]
df_clean_samp = df_merged[df_merged["plateAppearances"] >= 100].dropna(subset=["plateAppearances", "ops", "strikeOutRate", "walkRate", "pitchesPerPA","Total_ChallengeRate","Total_OverturnedRate"]).copy()
#print(df_clean_samp.head(5))
print(
    df_clean_samp[["plateAppearances", "ops", "strikeOutRate", "walkRate", "pitchesPerPA","Total_ChallengeRate","Total_OverturnedRate"]]
    .head(5)
    .to_markdown(index=False, floatfmt=".3f")
)

# Create function to run linear regression pipeline

In [ ]:
def evaluate_linear_regression_cv(df, target_col, target_label, min_pa=100):
  print(f"\n==================================================================")
  print(f"📈 5-FOLD CV LINEAR REGRESSION SUBSET SEARCH: {target_label.upper()}")
  print(f"==================================================================")

  # Filter out low-PA noise and drop missing values
  candidate_features = ["plateAppearances", "ops", "strikeOutRate", "walkRate", "pitchesPerPA"]
  df_clean = df[df["plateAppearances"] >= min_pa].dropna(subset=[target_col] + candidate_features).copy()

  # Strictly disjoint 80/20 train/test split
  X_full = df_clean[candidate_features]
  y_full = df_clean[target_col]
  X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, random_state=42
  )
  print(f"Disjoint Splits Created: Train = {X_train.shape} | Holdout Test = {X_test.shape}\n")

  # Configure 5-Fold Cross-Validation on the Training Set
  kf = KFold(n_splits=5, shuffle=True, random_state=42)
  cv_results = []

  # Test every possible combination of 1 to 5 features using 5-fold CV on X_train
  for r in range(1, len(candidate_features) + 1):
    for combo in itertools.combinations(candidate_features, r):
      cols = list(combo)
      model = LinearRegression()

      # Compute cross-validated R2 and RMSE across the 5 training folds
      cv_r2_scores = cross_val_score(model, X_train[cols], y_train, cv=kf, scoring="r2")
      cv_neg_mse_scores = cross_val_score(model, X_train[cols], y_train, cv=kf, scoring="neg_mean_squared_error")
      cv_rmse = np.sqrt(-cv_neg_mse_scores.mean())

      cv_results.append({
        "Features": ", ".join(cols),
        "Num_Features": len(cols),
        "CV_Mean_R2": cv_r2_scores.mean(),
        "CV_Std_R2": cv_r2_scores.std(),
        "CV_RMSE": cv_rmse,
        "Cols": cols
      })

  # Rank subsets by highest 5-Fold Cross-Validated Mean R2
  cv_df = pd.DataFrame(cv_results).sort_values(by="CV_Mean_R2", ascending=False).reset_index(drop=True)

  print("🏆 Top 5 Feature Combinations by 5-Fold CV Mean R-Squared (Training Set Only):")
  print(cv_df[["Features", "CV_Mean_R2", "CV_Std_R2", "CV_RMSE"]].head(5).to_string(index=False))

  # Select the single best feature combination from CV
  best_row = cv_df.iloc[0]
  best_cols = best_row["Cols"]

  # Fit the winning model on the entire training set and evaluate ONCE on the untouched holdout test set
  final_model = LinearRegression()
  final_model.fit(X_train[best_cols], y_train)
  y_test_pred = final_model.predict(X_test[best_cols])

  test_r2 = r2_score(y_test, y_test_pred)
  test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
  test_mae = mean_absolute_error(y_test, y_test_pred)

  print(f"\n✅ FINAL UNTOUCHED HOLDOUT TEST EVALUATION ({best_row['Features']}):")
  print(f"   • Holdout Test R²:   {test_r2:.4f}")
  print(f"   • Holdout Test RMSE: {test_rmse:.4f}")
  print(f"   • Holdout Test MAE:  {test_mae:.4f}")
  print(f"   • Intercept:         {final_model.intercept_:.6f}")
  for feat, coef in zip(best_cols, final_model.coef_):
    print(f"   • Coefficient [{feat}]: {coef:.6f}")

  # Plot Predicted vs. Actual Values on the Holdout Test Set
  plt.figure(figsize=(8, 6))
  sns.scatterplot(x=y_test, y=y_test_pred, alpha=0.7, edgecolor="black", s=60)

  min_val = min(y_test.min(), y_test_pred.min())
  max_val = max(y_test.max(), y_test_pred.max())
  plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", lw=2, label="Ideal 1:1 Fit")

  plt.title(f"Linear Regression: Predicted vs. Actual {target_label}\n(Holdout Test R² = {test_r2:.4f} | RMSE = {test_rmse:.4f})", fontweight="bold")
  plt.xlabel(f"Actual {target_label}")
  plt.ylabel(f"Predicted {target_label}")
  plt.legend()
  plt.grid(True, linestyle=":", alpha=0.6)
  plt.tight_layout()
  plt.show()

  return cv_df, final_model

# Run Pipeline to predict Batter ABS Challenge Rate

In [ ]:
# Predict Batter ABS Challenge Rate
challenge_rate_results = evaluate_linear_regression_cv(
  df=df_merged,
  target_col="Total_ChallengeRate",
  target_label="Batter ABS Challenge Rate",
  min_pa=100
)

# Run Pipeline to predict Batter ABS Overturn Rate

In [ ]:
# Predict Batter ABS Overturn Rate
overturn_rate_results = evaluate_linear_regression_cv(
  df=df_merged,
  target_col="Total_OverturnedRate",
  target_label="Batter ABS Overturn Rate",
  min_pa=100
)

# Create function to run logistic regression pipeline

In [ ]:
def evaluate_logistic_regression_cv(X, y, task_label, use_balancing=False):
  print(f"\n==================================================================")
  print(f"📊 5-FOLD CV LOGISTIC REGRESSION: {task_label.upper()}")
  print(f"==================================================================")

  # Strictly disjoint 80/20 stratified train/test split (matching Part 3 splits)
  X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
  )
  print(f"Disjoint Splits Created: Train = {X_train.shape} | Holdout Test = {X_test.shape}")

  weight_param = "balanced" if use_balancing else None

  # StandardScaler inside Pipeline prevents data leakage across CV folds
  pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(class_weight=weight_param, max_iter=1000, random_state=42))
  ])

  # Tune regularization strength C via 5-fold Stratified CV on training set only
  param_grid = {"logreg__C": [0.001, 0.01, 0.1, 1.0, 10.0]}
  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  scoring_metric = "f1_macro" if use_balancing else "accuracy"

  grid = GridSearchCV(pipe, param_grid, cv=cv, scoring=scoring_metric, n_jobs=-1)
  grid.fit(X_train, y_train)

  best_pipe = grid.best_estimator_
  best_c = grid.best_params_["logreg__C"]
  print(f"Best Hyperparameter (C) from 5-Fold CV: {best_c} (CV {scoring_metric}: {grid.best_score_:.4f})")

  # Evaluate ONCE on the untouched holdout test set
  y_pred = best_pipe.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  cm = confusion_matrix(y_test, y_pred)

  print(f"\n✅ FINAL UNTOUCHED HOLDOUT TEST EVALUATION:")
  print(f"   • Holdout Test Accuracy: {acc:.4f}")
  print("   • Confusion Matrix:")
  print(cm)
  print("\n   • Classification Report:")
  print(classification_report(y_test, y_pred))

  # Extract coefficients and Odds Ratios (exp(coef)) for plain-English interpretation
  logreg_model = best_pipe.named_steps["logreg"]
  # Note: coef_[0] represents log-odds for logreg_model.classes_[1] relative to classes_[0]
  pos_class = logreg_model.classes_[1]
  neg_class = logreg_model.classes_[0]
  coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Log_Odds_Coef": logreg_model.coef_[0],
    "Odds_Ratio": np.exp(logreg_model.coef_[0])
  }).sort_values(by="Log_Odds_Coef", key=abs, ascending=False)

  print(f"🔍 Coefficients & Odds Ratios for '{pos_class}' vs. '{neg_class}' (per +1 Std Dev):")
  print(coef_df.to_string(index=False))

  # Plot Confusion Matrix Heatmap
  plt.figure(figsize=(6, 5))
  sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
              xticklabels=logreg_model.classes_, yticklabels=logreg_model.classes_)
  plt.title(f"Logistic Regression: {task_label}\n(Holdout Test Acc = {acc:.4f})", fontweight="bold")
  plt.xlabel("Predicted Label")
  plt.ylabel("Actual Label")
  plt.tight_layout()
  plt.show()

  return best_pipe, coef_df

# Load data and feature engineering

In [ ]:
# Load pitch-by-pitch datasets from GitHub
batters_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Batters-FullSeason.csv"
defense_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Defense-FullSeason.csv"

df_batters = pd.read_csv(batters_url, encoding="latin1")
df_def = pd.read_csv(defense_url, encoding="latin1")

feature_cols = [
  "inning", "prePitchBalls", "prePitchStrikes", "outs", "challengesLeft",
  "pitchTypeGroup", "runnerOn1st", "runnerOn2nd", "runnerOn3rd",
  "plateTimeX", "plateTimeY_topDiff", "plateTimeY_BotDiff", "startSpeed", "netScore"
]

outcome_feature_cols = [
  "inning", "prePitchBalls", "prePitchStrikes", "outs", "challengesLeft",
  "pitchTypeGroup", "runnerOn1st", "runnerOn2nd", "runnerOn3rd",
  "startSpeed", "netScore"
]

# Remove rows with null pitch locations
df_batters = df_batters.dropna(subset=["plateTimeY", "plateTimeX"])

# Feature Engineering: Calculate vertical distance deviations from zone edges
# Positive values mean above the edge, negative values mean below the edge
df_batters["plateTimeY_topDiff"] = (
  df_batters["plateTimeY"] - df_batters["szTop"]
)
df_batters["plateTimeY_BotDiff"] = (
  df_batters["plateTimeY"] - df_batters["szBottom"]
)

# Set challengesLeft based on inningHalf (0 = Away, 1 = Home)
df_batters["challengesLeft"] = np.where(
  df_batters["inningHalf"] == 0,
  df_batters["awayChallengesLeft"],
  df_batters["homeChallengesLeft"],
)

# Set netScore based on inningHalf (0 = Away, 1 = Home)
# Positive if batting team is winning, 0 if tied, negative if losing
df_batters["netScore"] = np.where(
  df_batters["inningHalf"] == 0,
  df_batters["prePlayAwayScore"]-df_batters["prePlayHomeScore"],
  df_batters["prePlayHomeScore"]-df_batters["prePlayAwayScore"],
)

# Add pitchTypeGroup feature (1=fastballs, 2=breaking balls, 3=other)
# Define the conditions based on pitchTypeId
conditions = [
  df_batters["pitchTypeId"].isin([5, 6, 7, 12]),  # Fastballs
  df_batters["pitchTypeId"].isin([2, 3, 10, 13, 14, 15]),  # Breaking Balls
]

# Define the corresponding values for each condition
choices = [1, 2]

# Apply the conditions; default to 3 (Other) if no condition is met
df_batters["pitchTypeGroup"] = np.select(conditions, choices, default=3)

# Remove rows with null pitch locations
df_def = df_def.dropna(subset=["plateTimeY", "plateTimeX"])

# Feature Engineering: Calculate vertical distance deviations from zone edges
# Positive values mean above the edge, negative values mean below the edge
df_def["plateTimeY_topDiff"] = (
  df_def["plateTimeY"] - df_def["szTop"]
)
df_def["plateTimeY_BotDiff"] = (
  df_def["plateTimeY"] - df_def["szBottom"]
)

# Set challengesLeft based on inningHalf
df_def["challengesLeft"] = np.where(
  df_def["inningHalf"] == 1,
  df_def["awayChallengesLeft"],
  df_def["homeChallengesLeft"],
)

# Set netScore based on inningHalf
# Positive if defensive team is winning, 0 if tied, negative if losing
df_def["netScore"] = np.where(
  df_def["inningHalf"] == 1,
  df_def["prePlayAwayScore"]-df_def["prePlayHomeScore"],
  df_def["prePlayHomeScore"]-df_def["prePlayAwayScore"],
)

# Add pitchTypeGroup feature (1=fastballs, 2=breaking balls, 3=other)
# Define the conditions based on pitchTypeId
conditions = [
  df_def["pitchTypeId"].isin([5, 6, 7, 12]),  # Fastballs
  df_def["pitchTypeId"].isin([2, 3, 10, 13, 14, 15]),  # Breaking Balls
]

# Define the corresponding values for each condition
choices = [1, 2]

# Apply the conditions; default to 3 (Other) if no condition is met
df_def["pitchTypeGroup"] = np.select(conditions, choices, default=3)

# Run Logistic Regression pipeline to predict batter challenge intent

In [ ]:
# Batter Challenge Intent (Balanced due to ~95.6% Not Challenged skew)
df_b_clean = df_batters.dropna(subset=feature_cols + ["hasABSChallenge"]).copy()
X_b_intent = df_b_clean[feature_cols]
y_b_intent = df_b_clean["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})
print(
    X_b_intent[feature_cols]
    .head(5)
    .to_markdown(index=False, floatfmt=".3f")
)
evaluate_logistic_regression_cv(X_b_intent, y_b_intent, "Batter Challenge Intent (Balanced)", use_balancing=True)

# Run Logistic Regression pipeline to predict batter challenge outcome

In [ ]:
# Batter Challenge Outcome (Naturally balanced ~50/50)
df_b_chal = df_b_clean[df_b_clean["hasABSChallenge"] == 1].dropna(subset=["isOverturned"])
X_b_out = df_b_chal[outcome_feature_cols]
y_b_out = df_b_chal["isOverturned"].map({1: "Overturned", 0: "Confirmed"})
evaluate_logistic_regression_cv(X_b_out, y_b_out, "Batter Challenge Outcome", use_balancing=False)

# Run Logistic Regression pipeline to predict defense challenge intent

In [ ]:
# Defense Challenge Intent (Balanced due to ~97.6% Not Challenged skew)
df_d_clean = df_def.dropna(subset=feature_cols + ["hasABSChallenge"]).copy()
X_d_intent = df_d_clean[feature_cols]
y_d_intent = df_d_clean["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})
evaluate_logistic_regression_cv(X_d_intent, y_d_intent, "Defense Challenge Intent (Balanced)", use_balancing=True)

# Run Logistic Regression pipeline to predict defense challenge outcome

In [ ]:
# Defense Challenge Outcome (Naturally ~42/58)
df_d_chal = df_d_clean[df_d_clean["hasABSChallenge"] == 1].dropna(subset=["isOverturned"])
X_d_out = df_d_chal[outcome_feature_cols]
y_d_out = df_d_chal["isOverturned"].map({1: "Overturned", 0: "Confirmed"})
evaluate_logistic_regression_cv(X_d_out, y_d_out, "Defense Challenge Outcome", use_balancing=False)

# Create diagrams for website

In [ ]:
# Generate synthetic data modeled after Walk Rate vs. ABS Challenge Rate
np.random.seed(42)
x = np.array([0.04, 0.055, 0.07, 0.085, 0.10, 0.115, 0.13, 0.145, 0.16, 0.175])
true_slope = 0.22
true_intercept = 0.03
noise = np.array([0.008, -0.009, 0.011, -0.006, -0.012, 0.014, -0.007, 0.009, -0.010, 0.006])
y = true_intercept + true_slope * x + noise

# Fit OLS line of best fit
slope, intercept = np.polyfit(x, y, 1)
y_pred = intercept + slope * x

plt.figure(figsize=(8.5, 5.5))

# Plot dashed vertical residual lines
for i in range(len(x)):
  label = r"Residuals ($e_i = y_i - \hat{y}_i$)" if i == 0 else None
  plt.plot([x[i], x[i]], [y[i], y_pred[i]], color="crimson", linestyle="--", linewidth=1.6, zorder=2, label=label)

# Plot line of best fit
x_line = np.linspace(0.03, 0.185, 100)
y_line = intercept + slope * x_line
plt.plot(x_line, y_line, color="navy", linewidth=2.5, zorder=3,
         label=rf"Best-Fit Line ($\hat{{y}} = {intercept:.3f} + {slope:.2f}x$)")

# Plot actual observed data points
plt.scatter(x, y, color="blue", edgecolor="black", s=80, zorder=4, label=r"Observed Data Points ($y_i$)")

plt.title("Figure 1: Simple Linear Regression Best-Fit Line & Residual Errors", fontsize=13, fontweight="bold", pad=14)
plt.xlabel("Continuous Input Feature $X$ (e.g., Batter Walk Rate)", fontsize=11)
plt.ylabel("Continuous Target $Y$ (e.g., ABS Challenge Rate)", fontsize=11)
plt.xlim(0.03, 0.185)
plt.ylim(0.025, 0.085)
plt.legend(loc="upper left", frameon=True, shadow=True, fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig("lr_best_fit_residuals.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Generate continuous linear score z and pass through Sigmoid function
z = np.linspace(-6, 6, 400)
p = 1 / (1 + np.exp(-z))

# Sample points illustrating Class 0 vs Class 1 mapping
z_class0 = np.array([-4.8, -3.5, -2.4, -1.6, -0.8])
y_class0 = np.zeros_like(z_class0)

z_class1 = np.array([0.7, 1.5, 2.6, 3.8, 5.0])
y_class1 = np.ones_like(z_class1)

plt.figure(figsize=(8.5, 5.5))

# Shaded background decision zones
plt.axvspan(-6, 0, color="mistyrose", alpha=0.35, label="Predicted Class 0 Zone ($P < 0.5$)")
plt.axvspan(0, 6, color="palegreen", alpha=0.35, label=r"Predicted Class 1 Zone ($P \geq 0.5$)")

# Plot the Sigmoid S-curve
plt.plot(z, p, color="purple", linewidth=3, zorder=4,
         label=r"Sigmoid Function: $\sigma(z) = \frac{1}{1 + e^{-z}}$")

# Plot probability bounds (0 and 1) and 0.5 decision threshold
plt.axhline(y=1.0, color="gray", linestyle=":", linewidth=1.2)
plt.axhline(y=0.0, color="gray", linestyle=":", linewidth=1.2)
plt.axhline(y=0.5, color="black", linestyle="--", linewidth=1.5, label="Decision Threshold ($P = 0.5$)")
plt.axvline(x=0.0, color="black", linestyle="--", linewidth=1.5)

# Scatter actual binary outcome points along y=0 and y=1
plt.scatter(z_class0, y_class0, color="crimson", edgecolor="black", s=85, zorder=5, label="Actual Class 0 (e.g., Confirmed)")
plt.scatter(z_class1, y_class1, color="forestgreen", edgecolor="black", s=85, zorder=5, label="Actual Class 1 (e.g., Overturned)")

# Highlight the inflection point at (0, 0.5)
plt.scatter([0], [0.5], color="gold", edgecolor="black", s=100, zorder=6)
plt.annotate("Decision Boundary\n" + r"($z = 0 \Rightarrow P = 0.50$)",
             xy=(0, 0.5), xytext=(1.1, 0.36),
             arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
             fontsize=10, fontweight="bold")

plt.title("Figure 2: Logistic Regression Sigmoid Curve & Probability-to-Class Mapping", fontsize=13, fontweight="bold", pad=14)
plt.xlabel(r"Linear Log-Odds Score ($z = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n$)", fontsize=11)
plt.ylabel(r"Predicted Probability $P(Y = 1 \mid X)$", fontsize=11)
plt.xlim(-6, 6)
plt.ylim(-0.08, 1.08)
plt.legend(loc="center left", bbox_to_anchor=(0.01, 0.68), frameon=True, shadow=True, fontsize=9)
plt.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig("logreg_sigmoid_curve.png", dpi=300, bbox_inches="tight")
plt.show()